User will see the output after the generated output goes into self verification layer that checks whether it is correct or not . if yes then show output . it is a safety net 

In [20]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

load_dotenv("../.env")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.load_local(
    "../data/faiss_index", embedding_model,
    allow_dangerous_deserialization=True
)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print("Ready. Chunks:", vector_store.index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready. Chunks: 680


In [21]:
def verify_answer(context, answer):
    check_prompt = f"""You are a strict checker.

Context:
{context}

Answer:
{answer}

Is every fact in the answer supported by the context above?
Reply with only one word: SUPPORTED or NOT_SUPPORTED"""

    result = llm.invoke(check_prompt)
    text = result.content.strip().upper()
    return "NOT_SUPPORTED" not in text and "SUPPORTED" in text


In [22]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

THRESHOLD = 0.45

def ask(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best_score = results[0][1]
    print("Best score:", round(best_score, 2))

    if best_score < THRESHOLD:
        print("Answer: I don't have this information in the documents.")
        print("Action: Escalate to human (low confidence)")
        return

    pool = [doc for doc, score in results]
    pairs = [(question, d.page_content) for d in pool]
    r_scores = reranker.predict(pairs)
    ranked = sorted(zip(pool, r_scores), key=lambda x: x[1], reverse=True)
    docs = [d for d, s in ranked[:3]]
    print("Top chunk after rerank:", docs[0].metadata["title"])

    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {question}

Answer:"""

    answer = llm.invoke(prompt).content

    if answer.strip().startswith("I don't have this information"):
        print("\nAnswer:", answer)
        return

    if verify_answer(context, answer):
        print("\nVerification: PASSED")
        print("\nAnswer:", answer)
    else:
        print("\nVerification: FAILED - Try one more time")
        strict_prompt = prompt + "\nBe very strict: use ONLY facts written in the context. Keep the answer short."
        answer2 = llm.invoke(strict_prompt).content

        if verify_answer(context, answer2):
            print("Verification (retry): PASSED")
            print("\nAnswer:", answer2)
            answer = answer2
        else:
            print("Verification (retry): FAILED")
            print("\nAnswer: I don't have this information in the documents.")
            print("Action: Escalate to human (answer could not be verified)")
            return

    print("\nSources:")
    seen = set()
    for d in docs:
        if d.metadata["source"] not in seen:
            seen.add(d.metadata["source"])
            print("-", d.metadata["title"], "|", d.metadata["source"])


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [23]:
ask("How do I share a page with a guest?")
ask("How do I change my Notion plan?")


Best score: 0.67
Top chunk after rerank: Sharing & permissions settings in Notion

Verification: PASSED

Answer: To share a page with a guest:

1. **Select Share** at the top of the page.  
2. **Enter the guest’s email address** in the field where you add members or groups. (If the email is on one of your organization’s domains, the person will be added as a member instead of a guest.)  
3. **Open the dropdown** next to the entered email and choose the desired permission level for the guest.  
4. **Select Invite** to send the invitation.

**Make sure the following conditions are met before inviting a guest:**

- The guest’s email domain is allowed for your workspace.  
- Your workspace or teamspace permits guests to be added to pages.  
- Your workspace has not reached its guest‑limit.  

If any of these conditions are not satisfied, the invitation will not be successful.

Sources:
- Sharing & permissions settings in Notion | https://www.notion.com/en-gb/help/sharing-and-permissions
- 

In [24]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

question = "How do I invite someone from outside my company to a page?"
docs = vector_store.similarity_search(question, k=10)

pairs = [(question, d.page_content) for d in docs]
scores = reranker.predict(pairs)

ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
for d, s in ranked:
    print(f"{s:.2f} | {d.metadata['title']}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

5.74 | Sharing & permissions settings in Notion
5.11 | Intro to workspaces in Notion
3.85 | Sharing & permissions settings in Notion
2.02 | Sharing & permissions settings in Notion
1.91 | Sharing & permissions settings in Notion
-0.26 | Publish a website with Notion Sites
-0.62 | Sharing & permissions settings in Notion
-1.75 | Sharing & permissions settings in Notion
-5.62 | Sharing & permissions settings in Notion
-6.62 | Sharing & permissions settings in Notion
